<a href="https://colab.research.google.com/github/HLZHarry/Explainable_Alpha_Agent-Quality_Scorecard/blob/main/Initializing_the_Reasoning_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install
!pip install -q -U transformers accelerate bitsandbytes mcp langchain-community

In [ ]:
# 2. Mount Drive
# 2. Authenticate and Mount Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Check if we can see the drive
print(f"Drive mounted. Current directory: {os.getcwd()}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [ ]:
model_id  = "meta-llama/Meta-Llama-3-8B-Instruct"

In [ ]:
from huggingface_hub import login

# When you run this, it will ask for your token
login()

In [ ]:
# Configure 4-bit quatization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16
)

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,    # Reduces System RAM usage during loading
    torch_dtype=torch.bfloat16 # Ensures weights are handled in a smaller format from the start
)
print("Brain successfully initialized!")

## 1. `BitsAndBytesConfig`: The "Compressor" 📉

Standard AI models are massive. **Llama-3-8B**, for example, has 8 billion parameters. At 16-bit precision, this model requires ~**16GB of VRAM** just to load — often exceeding the limits of a free Colab GPU (typically 12–15GB).

`BitsAndBytesConfig` solves this with **4-bit Quantization:**

- **What it does:** Compresses 16-bit floats down to 4-bit integers
- **Why we use it:** Reduces memory footprint by nearly **75%**, allowing the model to fit into ~**5.5GB of VRAM** — leaving plenty of headroom for data and long-term alpha calculations

---

## 2. Why This Model? (`Llama-3-8B-Instruct`) 🧠

Three specific reasons drove this choice:

| # | Reason | Why It Matters |
|---|---|---|
| **1** | **Reasoning vs. Size** | Widely considered one of the "smartest" models for its size — punches well above its weight class in logic and instruction-following |
| **2** | **Instruction Following** | The `-Instruct` variant is fine-tuned to act as an assistant, enabling the specific **Chain-of-Thought** process our agent requires |
| **3** | **Ecosystem Support** | Its popularity means tools like `LangChain` and the `MCP SDK` have out-of-the-box support for its specific quirks |

In [ ]:
import os
import yfinance as yf
import pandas as pd

In [ ]:
# 1. Setup the project folder
project_folder = "/content/drive/MyDrive/Explainable_Alpha_Agent"
if not os.path.exists(project_folder):
    os.makedirs(project_folder)
    print(f"Created folder: {project_folder}")

In [ ]:
def save_ticker_to_drive(ticker_symbol):

  # 1. fetch Data
  ticker = yf.Ticker(ticker_symbol)
  bs = ticker.balance_sheet
  ic = ticker.financials
  cf = ticker.cashflow

  # 2. Save Raw Data
  file_name = f"{project_folder}/{ticker_symbol}_raw.csv"
  ic.transpose().to_csv(file_name)

  # 3. Perform Quality Calcualtions
  # Formula :ROIC = NOPAT / (Equity + Debt - Cash)
  try:
        ebit = ic.loc['EBIT'].iloc[0]
        tax_rate = 0.21 # Simplified corporate tax assumption
        nopat = ebit * (1 - tax_rate)

        equity = bs.loc['Stockholders Equity'].iloc[0]
        debt = bs.loc['Total Debt'].iloc[0] if 'Total Debt' in bs.index else 0
        cash = bs.loc['Cash And Cash Equivalents'].iloc[0]
        invested_capital = equity + debt - cash

        roic = (nopat / invested_capital) * 100

        return {
            "status": "Success",
            "ticker": ticker_symbol,
            "roic": f"{roic:.2f}%",
            "calculation_steps": f"NOPAT (${nopat/1e9:.1f}B) / Invested Capital (${invested_capital/1e9:.1f}B)",
            "file_stored_at": file_name
        }
  except Exception as e:
    return {"status": "Error", "message": str(e)}

# Let's test it with Visa (V)
analysis = save_ticker_to_drive("V")
print(analysis)

## The Definitions 📖

| Metric | Full Name | Definition |
|---|---|---|
| **EBIT** ⚙️ | Earnings Before Interest and Taxes | Also called *Operating Profit*. What remains after paying cost of goods and operating expenses (salaries, rent) — but before taxes or interest payments. Shows how hard the core business engine is working. |
| **NOPAT** 💸 | Net Operating Profit After Tax | A "what-if" number: *"If this company had no debt and paid a standard tax rate on its operating profit, how much cash would be left for owners?"* Calculated as $EBIT \times (1 - \text{Tax Rate})$. |
| **ROIC** 🎯 | Return on Invested Capital | The ultimate **Quality** metric. Measures how many cents of profit a company generates for every dollar of capital put to work. |

## Coding the Skill for Trends

In [ ]:
import numpy as np

In [ ]:
def calculate_roic_trajectory_robust(ticker_symbol):
    try:
        ticker = yf.Ticker(ticker_symbol)
        ic = ticker.financials
        bs = ticker.balance_sheet

        history = []
        # SAFETY CHECK: Only loop through years that actually exist in the data
        available_years = min(5, ic.shape[1], bs.shape[1])

        for i in range(available_years):
            ebit = ic.iloc[:, i].get('EBIT', 0)
            nopat = ebit * 0.79

            assets = bs.iloc[:, i].get('Total Assets', 0)
            cash = bs.iloc[:, i].get('Cash And Cash Equivalents', 0)
            liabilities = bs.iloc[:, i].get('Total Current Liabilities', 0)
            inv_cap = assets - cash - liabilities

            if inv_cap > 0:
                # Convert to standard float and round for the LLM
                roic = round(float((nopat / inv_cap) * 100), 2)
                history.append(roic)

        return {
            "ticker": ticker_symbol,
            "avg_roic": round(float(np.mean(history)), 2) if history else 0,
            "history": history,
            "status": "Success"
        }
    except Exception as e:
        return {"status": "Error", "message": str(e)}

In [ ]:
from mcp.server.fastmcp import FastMCP

In [ ]:
# Initialize our Server
mcp = FastMCP("ExplainableAlphaAgent")

In [ ]:
# Register the Trajectory Skill as a formal MCP Tool
@mcp.tool()
def calculate_roic_final(ticker_symbol):
    try:
        ticker = yf.Ticker(ticker_symbol)
        ic, bs = ticker.financials, ticker.balance_sheet
        history = []
        available_years = min(5, ic.shape[1], bs.shape[1])

        for i in range(available_years):
            ebit = ic.iloc[:, i].get('EBIT', 0)
            nopat = ebit * 0.79
            inv_cap = bs.iloc[:, i].get('Total Assets', 0) - bs.iloc[:, i].get('Cash And Cash Equivalents', 0) - bs.iloc[:, i].get('Total Current Liabilities', 0)
            if inv_cap > 0:
                history.append(round(float((nopat / inv_cap) * 100), 2))

        # KEY FIX: Flip the list so the LLM reads [Past -> Present]
        history.reverse()

        return {
            "ticker": ticker_symbol,
            "avg_roic": round(float(np.mean(history)), 2) if history else 0,
            "chronological_history": history,
            "status": "Success"
        }
    except Exception as e:
        return {"status": "Error", "message": str(e)}

## Agent Loop

## Connection Brain to Tools

In [ ]:
def ask_pension_agent(query):
    # Updated Prompt with Guardrails
    system_prompt = """You are a Senior Pension Fund Analyst.
    RULE 1: The 'Chronological History' is ordered from OLDEST to MOST RECENT.
    RULE 2: If the last number is higher than the first, the moat is IMPROVING.
    RULE 3: Show your math: Compare the oldest ROIC to the newest ROIC explicitly."""

    for ticker in ["V", "MSFT", "CVX"]:
        data = calculate_roic_final(ticker)
        prompt = f"{system_prompt}\n\nData: {data}\n\nAnalysis:"

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.2) # Low temp for accuracy
        print(tokenizer.decode(outputs[0], skip_special_tokens=True))
        print("\n" + "="*50 + "\n")

In [ ]:
# Test Full Agent
ask_pension_agent("Perform a moat audit.")

In [ ]:
def ask_pension_agent_v2(query):
    system_prompt = """You are a Senior Pension Fund Analyst at OMERS.
    Your job is to provide a 'Strategic Narrative' based on the Audit Data provided.
    Explain WHY the trend matters for a 20-year pension horizon."""

    for ticker in ["V", "MSFT", "CVX"]:
        data = calculate_roic_final(ticker) # Using our flipped [Past -> Present] logic

        # WE DO THE MATH IN PYTHON TO PREVENT HALLUCINATION
        oldest, newest = data['chronological_history'][0], data['chronological_history'][-1]
        trend = "IMPROVING" if newest > oldest else "FADING"
        change = round(newest - oldest, 2)

        # We feed the 'Truth' to the LLM so it can't lie
        prompt = f"""{system_prompt}
        Ticker: {ticker}
        Data: {data['chronological_history']}
        Fact: The moat is {trend} (Delta: {change} points).

        Provide the Strategic Narrative:"""

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.5)
        print(tokenizer.decode(outputs[0], skip_special_tokens=True))
        print("\n" + "="*50 + "\n")

ask_pension_agent_v2("Final Audit")

## 🏗️ Project Architecture: The "Explainable Alpha Agent"

> This is not just a chatbot — it is a **Federated AI Agent** built on a three-layer architecture designed for **Trust**, **Scale**, and **Explainability**.

---

### Layer 1: The Skill Layer — Deterministic Logic

- **The Problem:** LLMs are notoriously bad at arithmetic and can "hallucinate" financial trends when given raw lists of numbers.
- **The Solution:** A Python-based **Audit Skill** built with `yfinance` and `NumPy` that performs the heavy lifting — fetching 5 years of Balance Sheets and Income Statements, then calculating `ROIC` (Return on Invested Capital).
- **Key Innovation:** **Chronological Normalization** — flipping data to $[Oldest \rightarrow Newest]$ to align with the LLM's natural reading patterns, significantly reducing reasoning errors.

---

### Layer 2: The Bridge Layer — Model Context Protocol

- **The Standard:** Python logic wrapped using the **MCP (Model Context Protocol)**.
- **The Advantage:** Transforms "dead code" into a **Live Tool** that the LLM can discover and call on demand. In an enterprise setting like OMERS, this allows different teams (Risk, Equities, Infrastructure) to share a single **"Source of Truth"** for capital efficiency metrics.

---

### Layer 3: The Reasoning Layer — The "Pension-Style" Brain

- **The Logic:** A `Llama-3-8B` model (quantized for efficiency) with a System Prompt tuned to act as a **Senior Pension Analyst**.
- **The Focus:** Unlike retail trading bots that chase price momentum, this agent targets **Moat Durability** — identifying whether a company is a:
  - 📈 **"Compounding Machine"** *(e.g., Visa)* — sustained, improving capital efficiency
  - 📉 **"Fading Giant"** *(e.g., Chevron)* — declining multi-year efficiency trends